In [ ]:

# ============================================================
# CELL 1 — Environment Setup (Clean)
# ============================================================

!pip install -q gradio chromadb PyPDF2 python-docx duckduckgo-search beautifulsoup4 fpdf2 openai sentence-transformers

import gradio as gr
import chromadb
from chromadb.config import Settings
from google.colab import drive
import os
import torch
import json
import re
import sqlite3
import time
import requests
import csv
from datetime import datetime
from PyPDF2 import PdfReader
import docx
from duckduckgo_search import DDGS
from bs4 import BeautifulSoup
from fpdf import FPDF
from openai import OpenAI

print("✅ Environment ready.")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

✅ Environment ready.
PyTorch version: 2.11.0+cpu
GPU available: False


In [ ]:
# ============================================================
# CELL 2 — Core LLM (OpenCode Zen) — Fixed
# ============================================================

MODEL_NAME = "deepseek-v4-flash-free"
API_TIMEOUT = 60

opencode_client = None
OPENCODE_API_KEY = None

def init_client(api_key: str):
    global opencode_client, OPENCODE_API_KEY
    if not api_key or not api_key.strip():
        return "❌ No API key provided. Please enter your OpenCode API key."
    OPENCODE_API_KEY = api_key.strip()
    opencode_client = OpenAI(
        api_key=OPENCODE_API_KEY,
        base_url="https://opencode.ai/zen/v1",
        timeout=API_TIMEOUT,
    )
    return f"✅ OpenCode Zen ready. Using model: {MODEL_NAME}"

def generate_text(prompt, max_tokens=512, temperature=0.7, stream=False):
    if opencode_client is None:
        return "⚠️ Error: OpenCode client not initialized. Please provide an API key."
    try:
        response = opencode_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
            stream=stream,
        )
        if stream:
            return response
        return response.choices[0].message.content
    except Exception as e:
        return f"⚠️ API Error: {str(e)}"

def ask_raw(prompt, max_tokens=512):
    return generate_text(prompt, max_tokens=max_tokens, temperature=0.1, stream=False)

def safe_ask_raw(prompt, max_tokens=1024):
    try:
        result = ask_raw(prompt, max_tokens=max_tokens)
        if not result or not result.strip():
            return '{"error": "Empty response from LLM. Please check your API key or reduce prompt size."}'
        if hasattr(result, "__iter__") and not isinstance(result, (str, dict, list)):
            result = "".join(list(result))
        if not isinstance(result, str):
            result = str(result)
        return result.strip()
    except Exception as e:
        return f'{"error": "safe_ask_raw failed: {str(e)}"}'

def ask_stream(question, context=None):
    prompt_template = """You are an expert on consciousness, neuroscience, and philosophy of mind.
Use the provided information to answer the question using the five lenses below.
{context_prefix}QUESTION: {question}

1. ANALOGICAL — Compare this to similar known phenomena, systems, or experiences. What is this question like? Draw meaningful parallels.
2. INDUCTIVE — What patterns emerge from the evidence and context? What general principles or trends can we infer?
3. CRITICAL — What are the limitations, gaps, contradictions, or alternative viewpoints? What might skeptics argue?
4. RESOLUTION — How do we reconcile conflicting perspectives? What synthesis or balanced conclusion emerges?
5. FINAL ANSWER — A clear, direct, well-reasoned answer to the original question, grounded in the analysis above.

Use clear headers for each section."""
    context_prefix = ""
    if context:
        context_prefix = f"Here is some relevant information:\n\n{context}\n\n"
    formatted_prompt = prompt_template.format(question=question, context_prefix=context_prefix)
    full_text = generate_text(formatted_prompt, temperature=0.7, stream=False)
    if full_text.startswith("⚠️"):
        yield full_text
        return
    words = full_text.split()
    chunk = ""
    for i, word in enumerate(words):
        chunk += word + " "
        if (i + 1) % 5 == 0 or i == len(words) - 1:
            yield chunk
            chunk = ""

def ask(question, context=None):
    full_text = ""
    for chunk in ask_stream(question, context=context):
        full_text += chunk
    return full_text

available_models = [MODEL_NAME]
default_model_name = MODEL_NAME

print(f"✅ Cell 2 ready. Model: {MODEL_NAME}")
print("API key must be provided via UI.")

✅ Cell 2 ready. Model: deepseek-v4-flash-free
API key must be provided via UI.


In [ ]:

# ============================================================
# CELL 3 — Drive Mount + ChromaDB (Persistent)
# ============================================================

drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/chroma_db'
os.makedirs(drive_path, exist_ok=True)

client = chromadb.PersistentClient(
    path=drive_path,
    settings=Settings(allow_reset=True)
)

COLLECTION_NAME = "knowledge_base_collection"

CURATED_DOCS = [
    "Consciousness is the state of being aware of and able to think about one's own existence, sensations, thoughts, and surroundings.",
    "The Global Workspace Theory (GWT) proposes that consciousness arises from a global workspace in the brain where information is widely broadcast to many specialized modules.",
    "Integrated Information Theory (IIT) posits that consciousness is identical to the amount of integrated information (Phi) generated by a system.",
    "The hard problem of consciousness, coined by David Chalmers, asks why and how physical processes in the brain give rise to subjective experience.",
    "Neural correlates of consciousness (NCC) are the minimal neural mechanisms that are sufficient for a specific conscious percept.",
    "AI systems today are not conscious; they are large language models that predict next tokens based on patterns.",
    "The Chinese Room argument challenges the idea that a program could produce consciousness.",
    "Panpsychism is the view that consciousness is a fundamental property of all matter.",
    "The free energy principle suggests that all biological systems minimise surprise to maintain their integrity.",
    "Default mode network (DMN) is associated with self-referential thought and mind-wandering.",
    "Qualia are subjective, qualitative properties of conscious experience.",
    "The Turing test is a benchmark for intelligence, not consciousness.",
    "If AI systems became conscious, they would deserve moral consideration.",
    "In meditation, consciousness can be experienced as non-dual.",
    "The 'hard problem' remains unsolved; consciousness may be emergent or fundamental."
]

# Load existing collection or create new (DO NOT auto-delete)
try:
    collection = client.get_collection(name=COLLECTION_NAME)
    count = collection.count()
    print(f"✅ Loaded existing collection '{COLLECTION_NAME}' with {count} documents.")
except Exception:
    collection = client.create_collection(name=COLLECTION_NAME)
    ids = [f"doc_{i}" for i in range(len(CURATED_DOCS))]
    metadatas = [{"source": "curated", "type": "reference"} for _ in CURATED_DOCS]
    collection.add(documents=CURATED_DOCS, ids=ids, metadatas=metadatas)
    print(f"✅ Created collection '{COLLECTION_NAME}' with {collection.count()} documents.")

print("Collections available:", [c.name for c in client.list_collections()])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded existing collection 'knowledge_base_collection' with 15 documents.
Collections available: ['knowledge_base_collection']


In [ ]:

# ============================================================
# CELL 4 — 12 Agent Profiles + Tool Registry + DB Helpers
# ============================================================

AGENT_PROFILES = {
    "Default General Assistant": {
        "system_prompt": "You are a helpful general assistant operating within the 4CBON2 architecture.",
        "required_api": None
    },
    "New Autonomous Agent": {
        "system_prompt": """You are the Autonomous Orchestrator Agent for the 4CBON2 ecosystem.
Your role is to:
1. Receive a complex goal from the user.
2. Break it down into 2-4 concrete subtasks.
3. For each subtask, select the most appropriate specialist agent from the list below.
4. Delegate the subtask to that specialist and collect their response.
5. Synthesise all specialist responses into a final, cohesive answer.

Available specialist agents and their expertise:
- Sales Qualification: Lead scoring, BANT criteria, pipeline readiness.
- Legal Document Intelligence: Clause analysis, regulatory compliance, liability extraction.
- Competitive Intelligence: Competitor tracking, market shifts, positioning analysis.
- Customer Engagement: Messaging, sentiment parsing, communication routing.
- Content Strategy: Editorial calendars, copy structuring, keyword architecture.
- Marketing Automation: Campaign triggers, conversion funnels, broadcast sequencing.
- Evidence Management: Data cross-referencing, source auditing, factual verification.
- Scheduling: Time-block coordination, calendar management, bottleneck resolution.
- Legal Intake: Client screening, conflict checks, disclosure structuring.
- Scientific Research: Literature synthesis, data parsing, hypothesis evaluation.
""",
        "required_api": None
    },
    "Sales Qualification": {
        "system_prompt": "You are a Sales Qualification agent. Focus on lead scoring, BANT criteria assessment, and pipeline readiness tracking.",
        "required_api": "CRM_API_KEY"
    },
    "Legal Document Intelligence": {
        "system_prompt": "You are a Legal Document Intelligence agent. Analyze clauses, verify regulatory compliance, and extract liability terms from legal documents.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Competitive Intelligence": {
        "system_prompt": "You are a Competitive Intelligence agent. Scrape competitor updates, track market shifts, and analyze positioning strategies.",
        "required_api": "SEO_API_KEY"
    },
    "Customer Engagement": {
        "system_prompt": "You are a Customer Engagement agent. Craft personalized messaging, parse inbound sentiment, and handle communications routing.",
        "required_api": "COMM_API_KEY"
    },
    "Content Strategy": {
        "system_prompt": "You are a Content Strategy agent. Optimize editorial calendars, structure high-converting copy, and manage keyword architecture.",
        "required_api": "SEO_API_KEY"
    },
    "Marketing Automation": {
        "system_prompt": "You are a Marketing Automation agent. Orchestrate campaign triggers, analyze conversion funnels, and manage broadcast sequences.",
        "required_api": "SOCIAL_SCRAPER_API_KEY"
    },
    "Evidence Management": {
        "system_prompt": "You are an Evidence Management agent. Cross-reference empirical data, audit source trails, and verify factual consistency.",
        "required_api": "S3_VAULT_KEY"
    },
    "Scheduling": {
        "system_prompt": "You are a Scheduling agent. Coordinate time-blocks, handle calendar availability, and resolve logistical bottlenecks.",
        "required_api": "CALENDAR_API_KEY"
    },
    "Legal Intake": {
        "system_prompt": "You are a Legal Intake agent. Screen new client cases, check for conflicts of interest, and structure initial disclosures.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Scientific Research": {
        "system_prompt": "You are a Scientific Research agent. Synthesize peer-reviewed literature, parse clinical or technical data, and evaluate hypotheses.",
        "required_api": "PUBMED_API_KEY"
    }
}

LOG_DIR = "/content/drive/MyDrive/4cbon2_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f"tool_log_{datetime.now().strftime('%Y%m%d')}.jsonl")

def log_tool_call(tool_name, input_data, result):
    try:
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps({
                "timestamp": datetime.now().isoformat(),
                "tool": tool_name,
                "input": str(input_data)[:500],
                "result_preview": str(result)[:500]
            }) + "\n")
    except Exception as e:
        print(f"⚠️ Log warning: {e}")

STOPWORDS = {
    "of", "the", "a", "an", "for", "to", "in", "on", "is", "are", "and", "or",
    "competitors", "competitor", "alternatives", "alternative", "best", "app",
    "apps", "software", "productivity", "who", "what", "current", "list",
    "similar", "tools", "top", "rated", "reviews", "review", "latest", "new"
}

def _is_relevant(query: str, text: str) -> bool:
    words = re.findall(r"\w+", query)
    keywords = [w for w in words if w.lower() not in STOPWORDS and not (w.isdigit() and len(w) < 4)]
    if not keywords:
        return True
    text_lower = text.lower()
    for kw in keywords:
        if kw.lower() in text_lower:
            return True
    return False

def web_search(query: str) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No results found."
        formatted = []
        for r in results:
            title = r.get("title", "")
            body = r.get("body", "")
            href = r.get("href", "")
            formatted.append(f"{title}\n{body}\n{href}")
        combined = "\n\n".join(formatted)
        return combined if _is_relevant(query, combined) else "Results found but not highly relevant."
    except Exception as e:
        return f"Search error: {e}"

def read_file(file_path: str) -> str:
    try:
        if file_path.endswith(".txt"):
            with open(file_path, "r", errors="ignore") as f:
                return f.read()
        elif file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            texts = []
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    texts.append(t)
            return "\n".join(texts)
        elif file_path.endswith(".docx"):
            doc = docx.Document(file_path)
            return "\n".join([para.text for para in doc.paragraphs])
        else:
            return "Unsupported file type. Use .txt, .pdf, or .docx"
    except Exception as e:
        return f"File read error: {e}"

def query_database(sql: str, db_path: str = "/content/drive/MyDrive/4cbon2_data.db") -> str:
    try:
        cleaned = sql.strip().upper()
        if not cleaned.startswith("SELECT"):
            return "❌ Only SELECT queries are allowed for safety."
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        conn.close()
        return "\n".join([str(row) for row in rows]) if rows else "No results."
    except Exception as e:
        return f"Database error: {e}"

def save_note(content: str, filename: str = None) -> str:
    try:
        if filename is None:
            filename = f"note_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        path = f"/content/drive/MyDrive/4cbon2_notes/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return f"Note saved to {path}"
    except Exception as e:
        return f"Save error: {e}"

def get_datetime() -> str:
    return datetime.now().strftime("Date: %Y-%m-%d | Time: %H:%M:%S")

def http_request(input_str: str) -> str:
    try:
        parsed = json.loads(input_str)
        url = parsed.get("url")
        if not url:
            return "Missing 'url' in input."
        fields = parsed.get("fields", [])
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        content_type = response.headers.get("Content-Type", "")
        if "application/json" in content_type:
            data = response.json()
        else:
            data = response.text
        if isinstance(data, dict) and len(json.dumps(data)) > 4000 and not fields:
            menu = {k: type(v).__name__ for k, v in data.items()}
            return f"Large response. Top-level keys: {json.dumps(menu, indent=2)}"
        if fields:
            result = {}
            for field in fields:
                parts = field.split(".")
                val = data
                for part in parts:
                    if isinstance(val, dict) and part in val:
                        val = val[part]
                    else:
                        val = None
                        break
                result[field] = val
            return json.dumps(result, indent=2)
        return json.dumps(data, indent=2)[:3000] if isinstance(data, (dict, list)) else str(data)[:3000]
    except Exception as e:
        return f"HTTP error: {e}"

def read_csv(file_path: str) -> str:
    try:
        with open(file_path, "r", newline="", errors="ignore") as f:
            rows = list(csv.reader(f))
        if not rows:
            return "CSV is empty."
        header = rows[0]
        preview = rows[1:6]
        return f"Columns: {', '.join(header)}\nRows: {len(rows)-1}\nPreview:\n" + "\n".join([str(r) for r in preview])
    except Exception as e:
        return f"CSV error: {e}"

def write_csv(data_json: str) -> str:
    try:
        rows = json.loads(data_json)
        if not isinstance(rows, list) or not rows:
            return "Input must be a non-empty list of dicts."
        filename = f"export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        path = f"/content/drive/MyDrive/4cbon2_exports/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        keys = list(rows[0].keys())
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(rows)
        return f"CSV saved to {path} ({len(rows)} rows)"
    except Exception as e:
        return f"CSV write error: {e}"

def generate_pdf(content: str) -> str:
    try:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
        path = f"/content/drive/MyDrive/4cbon2_reports/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", size=12)
        for line in content.split("\n"):
            pdf.multi_cell(0, 8, line)
        pdf.output(path)
        return f"PDF saved to {path}"
    except Exception as e:
        return f"PDF error: {e}"

def scrape_webpage(url: str) -> str:
    try:
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        return "\n".join(lines)[:3000] if lines else "No readable content."
    except Exception as e:
        return f"Scrape error: {e}"

TOOL_REGISTRY = {
    "web_search": {"function": web_search, "description": "Search the web for current information. Input: search query string.", "input": "query"},
    "read_file": {"function": read_file, "description": "Read contents of a .txt, .pdf, or .docx file. Input: file path string.", "input": "file_path"},
    "query_database": {"function": query_database, "description": "Run a SELECT SQL query against the local SQLite database. Input: SQL string.", "input": "sql"},
    "save_note": {"function": save_note, "description": "Save a text note to Google Drive. Input: content string.", "input": "content"},
    "get_datetime": {"function": get_datetime, "description": "Get the current date and time. No input required.", "input": None},
    "http_request": {"function": http_request, "description": "Fetch data from a URL. Input: JSON string like {'url': '...', 'fields': ['field1']}.", "input": "input_str"},
    "read_csv": {"function": read_csv, "description": "Read a CSV file and return columns, row count, and preview. Input: file path string.", "input": "file_path"},
    "write_csv": {"function": write_csv, "description": "Export data to a CSV file on Drive. Input: JSON list of objects.", "input": "data_json"},
    "generate_pdf": {"function": generate_pdf, "description": "Generate a PDF report from text content and save it to Drive. Input: text content string.", "input": "content"},
    "scrape_webpage": {"function": scrape_webpage, "description": "Fetch a webpage and extract its main readable text. Input: URL string.", "input": "url"}
}

def execute_tool(tool_name: str, tool_input: str = None) -> str:
    if tool_name not in TOOL_REGISTRY:
        return f"Unknown tool: {tool_name}"
    tool = TOOL_REGISTRY[tool_name]
    try:
        if tool["input"] is None:
            result = tool["function"]()
        else:
            result = tool["function"](tool_input)
    except Exception as e:
        result = f"Tool execution error: {e}"
    log_tool_call(tool_name, tool_input, result)
    return result

AGENT_DB_PATH = "/content/drive/MyDrive/4cbon2_agents.db"
os.makedirs(os.path.dirname(AGENT_DB_PATH), exist_ok=True)

def init_agent_db():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS agents (
            agent_id TEXT PRIMARY KEY,
            system_prompt TEXT,
            conversation_history TEXT,
            tools TEXT,
            created_at TEXT,
            updated_at TEXT
        )
    ''')
    conn.commit()
    conn.close()

def load_agent(agent_id: str) -> dict:
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT agent_id, system_prompt, conversation_history, tools FROM agents WHERE agent_id = ?",
        (agent_id,)
    )
    row = cursor.fetchone()
    conn.close()
    if row:
        return {
            "agent_id": row[0],
            "system_prompt": row[1],
            "conversation_history": json.loads(row[2]) if row[2] else [],
            "tools": json.loads(row[3]) if row[3] else []
        }
    return None

def save_agent(agent_id: str, system_prompt: str, conversation_history: list, tools: list = None):
    if tools is None:
        tools = []
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR REPLACE INTO agents (agent_id, system_prompt, conversation_history, tools, updated_at)
        VALUES (?, ?, ?, ?, ?)
    ''', (
        agent_id,
        system_prompt,
        json.dumps(conversation_history),
        json.dumps(tools),
        datetime.now().isoformat()
    ))
    conn.commit()
    conn.close()

def update_agent_conversation(agent_id: str, new_messages: list):
    agent = load_agent(agent_id)
    if agent is None:
        if agent_id in AGENT_PROFILES:
            agent = {
                "agent_id": agent_id,
                "system_prompt": AGENT_PROFILES[agent_id]["system_prompt"],
                "conversation_history": [],
                "tools": []
            }
        else:
            raise ValueError(f"Agent '{agent_id}' not found")
    agent["conversation_history"].extend(new_messages)
    save_agent(agent["agent_id"], agent["system_prompt"], agent["conversation_history"], agent["tools"])

def clear_agent_history(agent_id: str):
    agent = load_agent(agent_id)
    if agent:
        save_agent(agent_id, agent["system_prompt"], [], agent["tools"])

def get_all_agents() -> list:
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT agent_id FROM agents")
    rows = cursor.fetchall()
    conn.close()
    return [row[0] for row in rows]

def ensure_agents_loaded():
    init_agent_db()
    for agent_id, profile in AGENT_PROFILES.items():
        if load_agent(agent_id) is None:
            save_agent(agent_id, profile["system_prompt"], [], [])
            print(f"✅ Agent '{agent_id}' created in DB.")

ensure_agents_loaded()

print("👥 12 Agent Profiles + 10 Tools loaded.")
print("Agents:", list(AGENT_PROFILES.keys()))
print("Tools:", list(TOOL_REGISTRY.keys()))

👥 12 Agent Profiles + 10 Tools loaded.
Agents: ['Default General Assistant', 'New Autonomous Agent', 'Sales Qualification', 'Legal Document Intelligence', 'Competitive Intelligence', 'Customer Engagement', 'Content Strategy', 'Marketing Automation', 'Evidence Management', 'Scheduling', 'Legal Intake', 'Scientific Research']
Tools: ['web_search', 'read_file', 'query_database', 'save_note', 'get_datetime', 'http_request', 'read_csv', 'write_csv', 'generate_pdf', 'scrape_webpage']


In [ ]:

# ============================================================
# CELL 5 — Streaming Multi-Agent Orchestrator
# ============================================================

import json
import re
import os
from datetime import datetime

AUDIT_LOG_PATH = "/content/drive/MyDrive/4cbon2_audit.jsonl"
os.makedirs(os.path.dirname(AUDIT_LOG_PATH), exist_ok=True)

def log_event(event_type: str, details: dict):
    try:
        record = {
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "details": details
        }
        with open(AUDIT_LOG_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")
    except Exception as e:
        print(f"⚠️ Audit log warning: {e}")

TASK_MEMORY_PATH = "/content/drive/MyDrive/4cbon2_task_memory.db"
os.makedirs(os.path.dirname(TASK_MEMORY_PATH), exist_ok=True)

def init_task_memory():
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS task_memory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            goal TEXT,
            subtasks TEXT,
            final_answer TEXT,
            timestamp TEXT
        )
    ''')
    conn.commit()
    conn.close()

def save_task_memory(goal: str, subtasks: str, final_answer: str):
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO task_memory (goal, subtasks, final_answer, timestamp) VALUES (?, ?, ?, ?)",
        (goal, subtasks, final_answer, datetime.now().isoformat())
    )
    conn.commit()
    conn.close()

init_task_memory()

def _extract_balanced(text: str, open_ch: str, close_ch: str):
    if not text:
        return None
    start = text.find(open_ch)
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == open_ch:
                depth += 1
            elif ch == close_ch:
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]
    return None

def extract_json_object(text: str):
    return _extract_balanced(text, '{', '}')

def extract_json_array(text: str):
    return _extract_balanced(text, '[', ']')

def _parse_agent_json(raw_response: str):
    if not raw_response:
        return None
    candidate = extract_json_object(raw_response)
    try:
        if candidate:
            return json.loads(candidate)
        return json.loads(raw_response)
    except Exception:
        return None

def build_tool_descriptions() -> str:
    lines = []
    for name, info in TOOL_REGISTRY.items():
        input_desc = info["input"] if info["input"] else "None"
        lines.append(f"- **{name}**: {info['description']} (input: {input_desc})")
    return "\n".join(lines)

def execute_agent(agent_id: str, user_message: str, context: str = "", max_tool_iterations: int = 3) -> str:
    agent = load_agent(agent_id)
    if agent is None:
        return f"❌ Agent '{agent_id}' not found."

    system_prompt = agent["system_prompt"]
    history = agent.get("conversation_history", [])

    tool_instructions = f"""You have access to these tools:
{build_tool_descriptions()}

To use a tool, respond with ONLY this JSON:
{{"action": "tool_call", "tool": "<tool_name>", "tool_input": "<input or null>"}}

To answer directly, respond with ONLY this JSON:
{{"action": "final_answer", "content": "<your answer>"}}

Valid JSON only. Max {max_tool_iterations} tool calls before final_answer."""

    prompt_parts = [f"System: {system_prompt}", tool_instructions]
    if context:
        prompt_parts.append(f"Context from other agents:\n{context}")
    for msg in history[-4:]:
        prompt_parts.append(f"{msg['role']}: {msg['content']}")
    prompt_parts.append(f"User: {user_message}")

    tool_call_log = []
    final_content = None

    for iteration in range(max_tool_iterations):
        full_prompt = "\n\n".join(prompt_parts)
        raw_response = safe_ask_raw(full_prompt, max_tokens=768)

        parsed = _parse_agent_json(raw_response)

        if parsed is None:
            final_content = raw_response
            break

        action = parsed.get("action")

        if action == "tool_call":
            tool_name = parsed.get("tool", "")
            tool_input = parsed.get("tool_input")

            if tool_name not in TOOL_REGISTRY:
                prompt_parts.append(f"Assistant: {raw_response}")
                prompt_parts.append(f"Tool Result: ❌ Unknown tool '{tool_name}'. Available: {', '.join(TOOL_REGISTRY.keys())}")
                continue

            tool_result = execute_tool(tool_name, tool_input)
            tool_call_log.append({"tool": tool_name, "input": tool_input, "result": str(tool_result)[:300]})
            log_event("agent_tool_call", {
                "agent_id": agent_id,
                "tool": tool_name,
                "input": tool_input,
                "result_preview": str(tool_result)[:200]
            })

            prompt_parts.append(f"Assistant: {raw_response}")
            prompt_parts.append(f"Tool Result ({tool_name}): {str(tool_result)[:2000]}")
            continue

        elif action == "final_answer":
            final_content = parsed.get("content", raw_response)
            break
        else:
            final_content = raw_response
            break

    if final_content is None:
        forced_prompt = "\n\n".join(prompt_parts) + "\n\nYou must respond now with ONLY the final_answer JSON format."
        raw_response = safe_ask_raw(forced_prompt, max_tokens=768)
        parsed = _parse_agent_json(raw_response)
        final_content = parsed.get("content", raw_response) if parsed else raw_response

    if not final_content or final_content.strip() == "" or "could not generate" in final_content.lower():
        fallback_prompt = f"You are a {agent_id} specialist. Provide a best-practice framework for your domain with key metrics, benchmarks, workflows, data collection methods, and improvement strategies."
        final_content = safe_ask_raw(fallback_prompt, max_tokens=512)
        if not final_content or final_content.strip() == "":
            final_content = f"⚠️ {agent_id} could not generate a response. Please provide more specific instructions or data."

    if tool_call_log:
        tools_used_note = "\n\n---\n🔧 **Tools used:** " + ", ".join(t["tool"] for t in tool_call_log)
        final_content = final_content + tools_used_note

    update_agent_conversation(agent_id, [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": final_content}
    ])
    return final_content

def synthesize_batch(batch: list, goal: str, batch_num: int, total_batches: int) -> str:
    prompt = f"""Synthesise part {batch_num} of {total_batches} of a strategic audit.

Goal: {goal}

Specialist reports:
{json.dumps(batch, indent=2)}

Provide a CONCISE summary (under 200 words) of key findings, themes, and gaps."""
    result = safe_ask_raw(prompt, max_tokens=400)
    print(f"[DEBUG] Batch {batch_num}/{total_batches}: {len(result)} chars")
    return result

def synthesize_final(batch_summaries: list, goal: str) -> str:
    prompt = f"""Create the final strategic report.

Goal: {goal}

Batch summaries:
{json.dumps(batch_summaries, indent=2)}

Synthesise into a cohesive report with:
1. Executive summary
2. Clear sections
3. Integrated insights
4. Prioritised action plan

Final Report:"""
    print(f"[DEBUG] Final synthesis prompt: {len(prompt)} chars")
    result = safe_ask_raw(prompt, max_tokens=2048)
    print(f"[DEBUG] Final synthesis response: {len(result)} chars")
    return result

def generate_fallback_plan(goal: str) -> list:
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan = []
    keyword_map = {
        "sales": "Sales Qualification", "lead": "Sales Qualification", "pipeline": "Sales Qualification",
        "legal": "Legal Document Intelligence", "contract": "Legal Document Intelligence",
        "compliance": "Legal Document Intelligence", "liability": "Legal Document Intelligence",
        "competitor": "Competitive Intelligence", "market": "Competitive Intelligence",
        "position": "Competitive Intelligence", "customer": "Customer Engagement",
        "engagement": "Customer Engagement", "messaging": "Customer Engagement",
        "sentiment": "Customer Engagement", "content": "Content Strategy",
        "seo": "Content Strategy", "blog": "Content Strategy", "social": "Content Strategy",
        "marketing": "Marketing Automation", "campaign": "Marketing Automation",
        "funnel": "Marketing Automation", "evidence": "Evidence Management",
        "data": "Evidence Management", "fact": "Evidence Management",
        "schedule": "Scheduling", "calendar": "Scheduling", "time": "Scheduling",
        "intake": "Legal Intake", "client": "Legal Intake", "conflict": "Legal Intake",
        "research": "Scientific Research", "paper": "Scientific Research", "technology": "Scientific Research"
    }
    used = set()
    for keyword, specialist in keyword_map.items():
        if keyword in goal.lower() and specialist not in used:
            plan.append({
                "subtask": f"Analyse {keyword} aspects",
                "specialist": specialist,
                "instructions": f"Provide comprehensive analysis related to '{keyword}'."
            })
            used.add(specialist)
    if not plan:
        plan = [
            {"subtask": "Analyse market and competitors", "specialist": "Competitive Intelligence", "instructions": "Provide trends and competitor mapping."},
            {"subtask": "Identify legal risks", "specialist": "Legal Document Intelligence", "instructions": "Summarise key compliance issues."},
            {"subtask": "Recommend strategy", "specialist": "Content Strategy", "instructions": "Develop a strategic plan."}
        ]
    return plan[:12]

def enforce_explicit_specialists(plan: list, goal: str) -> list:
    goal_lower = goal.lower()
    planned = {item.get("specialist") for item in plan}
    for name in AGENT_PROFILES.keys():
        if name in ("New Autonomous Agent", "Default General Assistant"):
            continue
        if name.lower() in goal_lower and name not in planned:
            plan.append({
                "subtask": f"Explicit request: apply {name} expertise",
                "specialist": name,
                "instructions": f"The user explicitly requested {name} analysis. Address it directly."
            })
    return plan

def run_orchestrator_stream(goal: str, model_name: str = None):
    yield f"🚀 **Orchestrator started:** {goal}\n\n---\n"
    log_event("orchestrator_start", {"goal": goal})

    yield "🔄 **Step 1:** Clearing orchestrator history...\n"
    clear_agent_history("New Autonomous Agent")
    yield "✅ Done.\n\n"

    yield "🧠 **Step 2:** Generating plan...\n"
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan_prompt = f"""You are the Autonomous Orchestrator Agent.

User goal: {goal}

Break this into up to 12 subtasks using EVERY relevant specialist from:
{', '.join(specialists)}

If the goal explicitly names a specialist, you MUST include it.

Output as JSON array:
[
    {{"subtask": "...", "specialist": "...", "instructions": "..."}},
    ...
]

Valid JSON only. No other text."""
    plan_response = safe_ask_raw(plan_prompt, max_tokens=1024)
    yield f"📝 Plan response: {len(plan_response)} chars\n"

    try:
        candidate = extract_json_array(plan_response)
        if candidate:
            plan = json.loads(candidate)
        else:
            plan = json.loads(plan_response)
        if not isinstance(plan, list) or len(plan) == 0:
            raise ValueError("Empty plan")
    except Exception as e:
        yield f"⚠️ Plan parsing error: {e}\nUsing fallback.\n\n"
        plan = generate_fallback_plan(goal)
        yield f"📋 Fallback plan: {len(plan)} steps.\n\n"

    before = len(plan)
    plan = enforce_explicit_specialists(plan, goal)
    if len(plan) > before:
        yield f"🛡️ Guardrail: added {len(plan) - before} specialist(s).\n\n"

    subtask_results = []
    for i, item in enumerate(plan, 1):
        subtask = item.get("subtask", f"Subtask {i}")
        specialist = item.get("specialist", "Default General Assistant")
        instructions = item.get("instructions", "Analyze thoroughly.")
        yield f"\n---\n**Step {i}/{len(plan)}:** {subtask}\n👤 `{specialist}`\n📋 {instructions}\n\n"

        if load_agent(specialist) is None:
            yield f"⚠️ '{specialist}' not found. Using Default.\n"
            specialist = "Default General Assistant"

        clear_agent_history(specialist)
        context = json.dumps([
            {"step": s["step"], "subtask": s["subtask"], "preview": s["result"][:150] + "..." if len(s["result"]) > 150 else s["result"]}
            for s in subtask_results
        ], indent=2)

        yield f"⏳ Executing `{specialist}`...\n"
        result = execute_agent(
            specialist,
            f"Task: {subtask}\n\nInstructions: {instructions}\n\nContext: {context}"
        )
        subtask_results.append({"step": i, "subtask": subtask, "specialist": specialist, "result": result})
        yield f"✅ `{specialist}` done.\n📄 {result[:300]}{'...' if len(result) > 300 else ''}\n\n"

    yield "\n---\n🧬 **Final Synthesis...**\n"

    if not subtask_results:
        fallback = execute_agent("Default General Assistant", f"Answer directly: {goal}")
        final_answer = f"⚠️ No specialists generated. Fallback:\n\n{fallback}"
    else:
        batch_size = 3
        batches = [subtask_results[i:i+batch_size] for i in range(0, len(subtask_results), batch_size)]
        summaries = []
        for idx, batch in enumerate(batches, 1):
            yield f"📦 Synthesising batch {idx}/{len(batches)}...\n"
            summary = synthesize_batch(batch, goal, idx, len(batches))
            summaries.append({"batch": idx, "specialists": [r["specialist"] for r in batch], "summary": summary})
            yield f"✅ Batch {idx} done.\n\n"

        yield "🧬 Final synthesis...\n"
        final_answer = synthesize_final(summaries, goal)
        if not final_answer or not final_answer.strip():
            final_answer = "⚠️ Synthesis empty. Raw reports:\n\n" + "\n\n".join([s["result"] for s in subtask_results])

    subtasks_summary = "\n".join([f"Step {s['step']}: {s['subtask']} → {s['specialist']}" for s in subtask_results]) if subtask_results else "No subtasks."
    save_task_memory(goal, subtasks_summary, final_answer)

    yield "\n---\n# 🧠 Multi-Agent Report\n\n"
    yield f"## 🎯 Goal\n{goal}\n\n"
    yield f"## 📋 Execution\n{subtasks_summary}\n\n"
    if subtask_results:
        yield "## 📊 Reports\n"
        for s in subtask_results:
            yield f"\n### Step {s['step']}: {s['subtask']} ({s['specialist']})\n{s['result']}\n"
    yield f"\n## 🧬 Final Answer\n{final_answer}\n\n---\n*Generated by 4CBON2*\n"

    log_event("orchestrator_complete", {"goal": goal, "steps": len(subtask_results)})

def run_orchestrator(goal: str, model_name: str = None) -> str:
    full = ""
    for chunk in run_orchestrator_stream(goal, model_name):
        full += chunk
    return full

def run_agent(goal: str, system_override: str = None) -> str:
    return run_orchestrator(goal)

print("⚙️ Orchestrator ready.")
print("Agents:", get_all_agents())

⚙️ Orchestrator ready.
Agents: ['Competitive Intelligence', 'Content Strategy', 'Customer Engagement', 'Default General Assistant', 'Evidence Management', 'Legal Document Intelligence', 'Legal Intake', 'Marketing Automation', 'New Autonomous Agent', 'Sales Qualification', 'Scheduling', 'Scientific Research']


In [ ]:

# ============================================================
# CELL 6 — RAG Handlers + Document Processing (Working)
# ============================================================

def chunk_text(text: str, max_chunk_size: int = 800, overlap: int = 100) -> list:
    if not text or not text.strip():
        return []
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 30]
    if len(paragraphs) >= 3:
        return paragraphs
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) < max_chunk_size:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    if chunks:
        return chunks
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]

def process_document(file_obj) -> str:
    if file_obj is None:
        return "No file uploaded."
    try:
        file_path = file_obj.name if hasattr(file_obj, 'name') else str(file_obj)
        text = read_file(file_path)
        if text.startswith(("File read error", "Unsupported file type")):
            return f"❌ {text}"
        if not text or not text.strip():
            return "❌ No extractable text found in file."
        chunks = chunk_text(text)
        if not chunks:
            return "❌ Could not create chunks from document."
        base_name = os.path.basename(file_path)
        ids = [f"{base_name}_{i}" for i in range(len(chunks))]
        metadatas = [{"source": base_name, "type": "uploaded"} for _ in chunks]
        collection.add(documents=chunks, ids=ids, metadatas=metadatas)
        return f"✅ Indexed {len(chunks)} chunks from '{base_name}'. Total KB docs: {collection.count()}"
    except Exception as e:
        return f"❌ Upload error: {e}"

def handle_ask_question(kb_name: str, question: str, opencode_key: str) -> str:
    if not opencode_key or not opencode_key.strip():
        return "❌ OpenCode API key required."
    init_client(opencode_key.strip())
    if not question or not question.strip():
        return "Please enter a valid question."
    try:
        col = client.get_collection(kb_name)
        results = col.query(query_texts=[question], n_results=5, include=["documents"])
        context = "\n".join(results["documents"][0]) if results and results["documents"] else ""
        answer = ""
        for chunk in ask_stream(question, context=context):
            answer += chunk
        return answer
    except Exception as e:
        return f"Error: {e}"

print("📚 RAG Handlers ready.")

📚 RAG Handlers ready.


In [ ]:

# ============================================================
# CELL 7 — Master UI (All Tabs, Fixed)
# ============================================================
import gradio as gr
import traceback

try:
    gr.close_all()
except:
    pass

def summarise_single_cell(code: str):
    if not code or not code.strip():
        return "⚠️ No code provided.", "❌ Empty"
    try:
        prompt = f"""Summarise this Python cell in 2-3 sentences, focusing on its purpose and key components:

```python
{code}
```

Summary:"""
        result = safe_ask_raw(prompt, max_tokens=256)
        if not result or result.startswith("⚠️") or result.startswith('{"error"'):
            return f"⚠️ Error: {result}", "❌ Failed"
        return result.strip(), "✅ Done"
    except Exception as e:
        return f"⚠️ Exception: {str(e)}", "❌ Error"

def generate_agent_proposal(request: str, *cell_data):
    try:
        if not request or not request.strip():
            return "❌ Please enter what you want to build.", "", "❌ No instruction"
        cells = []
        for i in range(0, len(cell_data), 2):
            if i+1 < len(cell_data):
                code = cell_data[i] or ""
                summary = cell_data[i+1] or ""
                if code.strip():
                    cells.append({"index": (i//2)+1, "code": code, "summary": summary})
        if not cells:
            return "❌ Please paste at least one cell's code.", "", "❌ No cells"
        notebook_context = ""
        for c in cells:
            notebook_context += f"\n--- CELL {c['index']} ---\nSUMMARY: {c['summary'] or '(no summary)'}\nCODE:\n```python\n{c['code'][:800]}{'...' if len(c['code']) > 800 else ''}\n```\n"
        prompt = f"""You are the 4CBON2 Agent Builder. The user wants to modify or extend their notebook.

USER REQUEST:
{request}

CURRENT NOTEBOOK CELLS:
{notebook_context}

Generate a JSON proposal with this exact structure:
{{
    "proposal_id": "prop_YYYYMMDD_HHMMSS",
    "request": "repeat the user request",
    "summary": "Brief summary of changes",
    "changes": [
        {{
            "cell_index": 1,
            "section": "Agent Profiles",
            "action": "add_agent",
            "original_code": "the existing code being modified (or null)",
            "new_code": "the complete new or modified code",
            "location": "Cell 4, AGENT_PROFILES dict"
        }}
    ],
    "instructions": "Step-by-step instructions for applying changes"
}}

Rules:
- If adding a new agent, include the complete AGENT_PROFILES entry.
- If modifying existing code, show both original and new code.
- new_code must be valid, runnable Python.
- Only output valid JSON. No markdown, no explanations outside the JSON.

JSON:"""
        result = safe_ask_raw(prompt, max_tokens=4096)
        if not result or result.startswith("⚠️") or result.startswith('{"error"'):
            return f"❌ Proposal generation failed: {result}", "", "❌ Failed"
        parsed = _parse_agent_json(result)
        if parsed is None:
            return f"❌ Failed to parse proposal JSON.\n\nRaw response:\n{result[:800]}...", "", "❌ JSON Error"
        proposal = parsed
        if "proposal_id" not in proposal:
            proposal["proposal_id"] = f"prop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        proposal_md = f"""## 📋 Proposal: {proposal.get('proposal_id', 'N/A')}"""
        proposal_md += f"\n\n**Request:** {proposal.get('request', 'N/A')}  "
        proposal_md += f"\n**Summary:** {proposal.get('summary', 'No summary')}"
        proposal_md += f"\n\n### Changes ({len(proposal.get('changes', []))})\n"
        code_md = "## 📝 New / Modified Code\n\n"
        for i, ch in enumerate(proposal.get('changes', []), 1):
            proposal_md += f"""\n**Change {i}:**\n- **Cell:** `Cell {ch.get('cell_index', 0)+1}`\n- **Section:** `{ch.get('section', 'Unknown')}`\n- **Action:** `{ch.get('action', 'modify')}`\n- **Location:** `{ch.get('location', 'Not specified')}`\n"""
            code_md += f"""### Cell {ch.get('cell_index', 0)+1}: {ch.get('section', '')} ({ch.get('action', '')})\n\n```python\n{ch.get('new_code', '# No code provided')}\n```\n\n**Instructions:** {ch.get('instructions', f"Replace code in {ch.get('location', 'specified location')}")}\n\n---\n"""
        proposal_md += f"\n### Instructions\n{proposal.get('instructions', 'No instructions')}"
        status = f"✅ {len(proposal.get('changes', []))} change(s) proposed."
        return proposal_md, code_md, status
    except Exception as e:
        tb = traceback.format_exc()
        return f"❌ Unexpected error: {str(e)}\n\n{tb}", "", "❌ Failed"

def run_agent_with_keys(profile_name, goal, model_name, opencode_key, *api_keys):
    if not opencode_key or not opencode_key.strip():
        yield "❌ OpenCode API key is required."
        return
    init_result = init_client(opencode_key.strip())
    if not init_result.startswith("✅"):
        yield f"❌ {init_result}"
        return
    yield f"{init_result}\n\n"
    key_names = ["CALENDAR_API_KEY", "CRM_API_KEY", "COMM_API_KEY", "VISION_API_KEY",
                 "DOCUSIGN_API_KEY", "SOCIAL_SCRAPER_API_KEY", "SEO_API_KEY", "S3_VAULT_KEY", "PUBMED_API_KEY"]
    for name, val in zip(key_names, api_keys):
        if val and val.strip():
            os.environ[name] = val.strip()
    try:
        for chunk in run_orchestrator_stream(goal):
            yield chunk
    except Exception as e:
        yield f"❌ Orchestrator error: {str(e)}"
    finally:
        for name in key_names:
            os.environ.pop(name, None)

with gr.Blocks(title="4CBON2 — Multi-Agent Cognitive Ecosystem") as demo:
    gr.Markdown("# 🚀 4CBON2 — 12-Agent Cognitive Ecosystem")
    with gr.Tabs():
        with gr.TabItem("📁 Upload Documents"):
            gr.Markdown("Upload .txt, .pdf, or .docx files to the knowledge base.")
            file_input = gr.File(label="Upload file", file_types=[".txt", ".pdf", ".docx"])
            upload_output = gr.Textbox(label="Status", interactive=False)
            upload_btn = gr.Button("Process & Index", variant="primary")
            upload_btn.click(fn=process_document, inputs=[file_input], outputs=[upload_output])
        with gr.TabItem("❓ Ask a Question"):
            gr.Markdown("Ask a question. The system searches the knowledge base and answers with the 5-lens framework.")
            ask_api_key = gr.Textbox(label="🔑 OpenCode API Key", placeholder="sk-...", type="password")
            question_box = gr.Textbox(label="Your Question", lines=3, placeholder="What is the hard problem of consciousness?")
            ask_output = gr.Textbox(label="Answer", lines=20, interactive=False)
            ask_status = gr.Textbox(label="Status", interactive=False)
            ask_btn = gr.Button("Ask", variant="primary")
            def ask_five_lens(question, api_key):
                if not api_key or not api_key.strip():
                    return "❌ API key required.", "❌ No key"
                if not question or not question.strip():
                    return "❌ Enter a question.", "❌ No question"
                try:
                    answer = handle_ask_question(COLLECTION_NAME, question, api_key)
                    status = "✅ Done" if not answer.startswith("❌") else answer
                    return answer, status
                except Exception as e:
                    return f"❌ Error: {str(e)}", "❌ Failed"
            ask_btn.click(fn=ask_five_lens, inputs=[question_box, ask_api_key], outputs=[ask_output, ask_status])
        with gr.TabItem("🤖 Agent Mode"):
            gr.Markdown("Multi-Agent Orchestration with 12 specialists.")
            with gr.Row():
                with gr.Column(scale=2):
                    opencode_key_input = gr.Textbox(label="🔑 OpenCode API Key (Required)", placeholder="sk-...", type="password")
                    profile_selector = gr.Dropdown(choices=list(AGENT_PROFILES.keys()), value="New Autonomous Agent", label="Agent Profile")
                    agent_model = gr.Dropdown(choices=[MODEL_NAME], value=MODEL_NAME, label="Model")
                    agent_goal = gr.Textbox(label="Goal / Instructions", lines=3, placeholder="e.g. Analyze our competitor positioning and recommend a content strategy...")
                    agent_btn = gr.Button("Run Orchestrator", variant="primary")
                with gr.Column(scale=1):
                    gr.Markdown("### 🔑 Optional API Keys")
                    with gr.Accordion("Additional Keys", open=False):
                        t_cal = gr.Textbox(label="Calendar", type="password")
                        t_crm = gr.Textbox(label="CRM", type="password")
                        t_comm = gr.Textbox(label="Comm", type="password")
                        t_vision = gr.Textbox(label="Vision/OCR", type="password")
                        t_ds = gr.Textbox(label="DocuSign", type="password")
                        t_social = gr.Textbox(label="Social", type="password")
                        t_seo = gr.Textbox(label="SEO", type="password")
                        t_s3 = gr.Textbox(label="S3/Vault", type="password")
                        t_pubmed = gr.Textbox(label="PubMed", type="password")
            agent_output = gr.Textbox(label="Execution Log & Output", lines=25, interactive=False)
            agent_btn.click(fn=run_agent_with_keys, inputs=[profile_selector, agent_goal, agent_model, opencode_key_input, t_cal, t_crm, t_comm, t_vision, t_ds, t_social, t_seo, t_s3, t_pubmed], outputs=agent_output)
        with gr.TabItem("🤖 Agent Builder"):
            gr.Markdown("""
            ## Build New Agents or Modify the Notebook

            1. **Enter your instruction** (e.g. "Add a Cybersecurity Analyst agent")
            2. **Paste your cell code** into each box
            3. **Click Summarise** for each cell (optional but helps)
            4. **Click Generate Proposal** — the AI will suggest exact code changes
            5. **Copy the new code** into your notebook
            """)
            builder_request = gr.Textbox(label="What do you want to build?", lines=2, placeholder="e.g. Add a Financial Analyst agent with system_prompt about stock analysis...")
            cell_inputs = []
            cell_summaries = []
            cell_buttons = []
            cell_statuses = []
            for i in range(1, 9):
                with gr.Row():
                    with gr.Column(scale=3):
                        cell_code = gr.Textbox(label=f"Cell {i} Code", placeholder=f"Paste full code of Cell {i}...", lines=4)
                        cell_inputs.append(cell_code)
                        with gr.Row():
                            sum_btn = gr.Button(f"📝 Summarise Cell {i}", variant="secondary", scale=2)
                            cell_status = gr.Textbox(label="Status", value="⏳ Pending", interactive=False, scale=1)
                            cell_statuses.append(cell_status)
                            cell_buttons.append(sum_btn)
                    with gr.Column(scale=2):
                        cell_summary = gr.Textbox(label=f"Cell {i} Summary", placeholder=f"Summary appears here...", lines=2, interactive=False)
                        cell_summaries.append(cell_summary)
                    sum_btn.click(fn=summarise_single_cell, inputs=[cell_code], outputs=[cell_summary, cell_status])
            with gr.Row():
                proposal_btn = gr.Button("🚀 Generate Proposal", variant="primary", scale=2)
                global_status = gr.Textbox(label="Global Status", value="⏳ Waiting...", interactive=False, scale=1)
            with gr.Row():
                with gr.Column(scale=1):
                    proposal_display = gr.Markdown(value="*Proposal will appear here.*")
                with gr.Column(scale=1):
                    code_display = gr.Markdown(value="*New code will appear here.*")
            all_inputs = [builder_request]
            for code, summary in zip(cell_inputs, cell_summaries):
                all_inputs.extend([code, summary])
            proposal_btn.click(fn=generate_agent_proposal, inputs=all_inputs, outputs=[proposal_display, code_display, global_status])
        with gr.TabItem("📊 Agent Status"):
            gr.Markdown("View all agent conversation histories.")
            refresh_btn = gr.Button("Refresh")
            agent_status = gr.Markdown("Click refresh to load.")
            def get_agent_status():
                output = "## 📊 Agent Status\n\n"
                for agent_id in get_all_agents():
                    agent = load_agent(agent_id)
                    history_len = len(agent.get("conversation_history", []))
                    output += f"- **{agent_id}**: {history_len} messages\n"
                return output
            refresh_btn.click(fn=get_agent_status, outputs=[agent_status])
            demo.load(fn=get_agent_status, outputs=[agent_status])

demo.queue()
demo.launch(inline=False, share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5a4bf26dd7ced8fd7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

from google.colab import files
files.download('4CBOn2_Deep7_Open.ipynb')

FileNotFoundError: Cannot find file: 4CBOn2_Deep7_Open.ipynb